In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

# =========================
# PATHS
# =========================
LANDING_PATH = "/Volumes/workspace/final_project/landing/"

ORDERS_PATH = LANDING_PATH + "orders.json"
INCIDENTS_PATH = LANDING_PATH + "incidents.csv"
SUPPLIERS_PATH = LANDING_PATH + "suppliers.parquet"

# =========================
# READ RAW FILES
# =========================
orders = (
    spark.read
    .option("multiline", "true")
    .json(ORDERS_PATH)
)

incidents = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(INCIDENTS_PATH)
)

suppliers = (
    spark.read
    .parquet(SUPPLIERS_PATH)
)
# =========================
# CHECK SCHEMA
# =========================
print("=== ORDERS SCHEMA ===")
orders.printSchema()

print("=== INCIDENTS SCHEMA ===")
incidents.printSchema()

print("=== SUPPLIERS SCHEMA ===")
suppliers.printSchema()

# =========================
# PREVIEW
# =========================
display(orders.limit(10))
display(incidents.limit(10))
display(suppliers.limit(10))



# =========================
# WRITE DELTA TABLES
# =========================
orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.final_project.bronze_orders")

incidents.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.final_project.bronze_incidents")

suppliers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.final_project.bronze_suppliers")

# =========================
# FINAL CHECK
# =========================
print("=== BRONZE TABLES CREATED ===")

display(spark.table("workspace.final_project.bronze_orders").limit(10))
display(spark.table("workspace.final_project.bronze_incidents").limit(10))
display(spark.table("workspace.final_project.bronze_suppliers").limit(10))